# NB02 — Tokenisation, PyTorch Dataset et Split 90/5/5
> **Phase B** : Préparation des données pour l'apprentissage statistique.


In [ ]:
!pwd
!ls /content
!git clone -b b/s1-portability https://github.com/AhmadBelhaj4/Recherche_AI.git /content/Recherche_AI
!ls /content/Recherche_AI

In [ ]:
# ==============================================================================
# Bootstrap : détection du répertoire racine (local ET Google Colab)
# ==============================================================================
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/AhmadBelhaj4/Recherche_AI.git"
BRANCH = "b/s1-portability"   # test only; set back to "main" after the merge


def _find_root(start):
    """Walk up from `start` looking for utils.py (also <p>/project1/utils.py)."""
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / "utils.py").exists():
            return p
        if (p / "project1" / "utils.py").exists():
            return p / "project1"
    return None


ROOT_DIR = _find_root(Path.cwd())

if ROOT_DIR is None and Path("/content").exists():      # Colab
    clone_target = Path("/content/Recherche_AI")
    if not clone_target.exists():
        subprocess.run(
            ["git", "clone", "-b", BRANCH, REPO_URL, str(clone_target)],
            check=True,
        )
    ROOT_DIR = _find_root(clone_target)

if ROOT_DIR is None:
    raise FileNotFoundError(
        "Could not locate utils.py. Open the notebook inside the Recherche_AI "
        "repo, or on Colab check that the git clone succeeded."
    )

sys.path.insert(0, str(ROOT_DIR))
from utils import get_paths
PATHS = get_paths()
print(f"ROOT_DIR  : {ROOT_DIR}")
print(f"PATHS     : {PATHS}")

In [ ]:
# ==============================================================================
# Cellule 1 : Paramètres généraux (Settings) et reproductibilité
# ==============================================================================
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from utils import set_seed, AminoAcidTokenizer, SequenceDataset, collate_fn, make_loader, shift

SEED = 42
set_seed(SEED)

BATCH_SIZE = 64
MIN_LEN = 100
MAX_LEN = 140
SPLIT_RATIOS = (0.90, 0.05, 0.05)

# DATA_PATH: prefer processed vhh_clean.csv, fall back to toy data
DATA_PATH = PATHS["processed"] / "vhh_clean.csv"
if not DATA_PATH.exists():
    DATA_PATH = PATHS["raw"].parent / "toy" / "vhh_clean.csv"

print(f"Seed: {SEED} | Batch size: {BATCH_SIZE} | Source: {DATA_PATH}")


In [ ]:
# ==============================================================================
# Cellule 2 : Tokeniseur par caractère (Alphabet des 20 acides aminés)
# ==============================================================================
tokenizer = AminoAcidTokenizer()
print(f"Taille du vocabulaire : {tokenizer.vocab_size} tokens")
print(f"Tokens : {tokenizer.aa_to_id}")

test_seq = "QVQLVESGGGLVQPGGSLRLSCAASGFTFS"
encoded = tokenizer.encode(test_seq, add_special_tokens=True)
decoded = tokenizer.decode(encoded)
print(f"Test encodage : {encoded}")
print(f"Test décodage : {decoded}")
assert decoded == test_seq, "Erreur dans le tokeniseur !"


In [ ]:
# ==============================================================================
# Cellule 3 : SequenceDataset et DataLoader avec Collate
# ==============================================================================
import subprocess, sys
_toy = PATHS["raw"].parent / "toy" / "vhh_clean.csv"
if not DATA_PATH.exists() and not _toy.exists():
    subprocess.run([sys.executable, str(ROOT_DIR / "scripts" / "make_toy_data.py")], check=True)
df = pd.read_csv(DATA_PATH)
sequences = df["sequence"].tolist()
print(f"Séquences chargées : {len(sequences):,}")

tokenizer = AminoAcidTokenizer()
loader = make_loader(sequences, tokenizer, batch_size=BATCH_SIZE, shuffle=True)

batch = next(iter(loader))
inputs, targets = shift(batch)
print(f"Batch inputs  : {inputs.shape} [dtype: {inputs.dtype}]")
print(f"Batch targets : {targets.shape} [dtype: {targets.dtype}]")


In [ ]:
# ==============================================================================
# Cellule 4 : Split aléatoire 90% / 5% / 5% (Plan du professeur)
# ==============================================================================
n = len(sequences)
indices = np.random.permutation(n)
n_train = int(SPLIT_RATIOS[0] * n)
n_val = int(SPLIT_RATIOS[1] * n)

train_seqs = [sequences[i] for i in indices[:n_train]]
val_seqs = [sequences[i] for i in indices[n_train:n_train + n_val]]
test_seqs = [sequences[i] for i in indices[n_train + n_val:]]

print(f"Train : {len(train_seqs):,} ({len(train_seqs)/n:.1%})")
print(f"Val   : {len(val_seqs):,} ({len(val_seqs)/n:.1%})")
print(f"Test  : {len(test_seqs):,} ({len(test_seqs)/n:.1%})")

# Vérification stricte de disjonction
assert set(train_seqs).isdisjoint(set(val_seqs)), "Fuite entre train et val !"
assert set(train_seqs).isdisjoint(set(test_seqs)), "Fuite entre train et test !"
assert set(val_seqs).isdisjoint(set(test_seqs)), "Fuite entre val et test !"
print("Vérification : Les trois ensembles sont strictement disjoints.")

out_dir = PATHS["raw"].parent
out_dir.mkdir(exist_ok=True)
pd.DataFrame({"sequence": train_seqs}).to_csv(out_dir / "train.csv", index=False)
pd.DataFrame({"sequence": val_seqs}).to_csv(out_dir / "val.csv", index=False)
pd.DataFrame({"sequence": test_seqs}).to_csv(out_dir / "test.csv", index=False)
print("Splits sauvegardés dans data/.")
